# Lexora — resume MuRIL fine-tuning on Kaggle
Keep this notebook and its input dataset PRIVATE. Enable Internet and GPU T4 x2.
Run using Save Version > Save & Run All. This uses one GPU and preserves the original five-fold experiment.
Completed folds are reused. Epoch snapshots and results stay in /kaggle/working/muril-run.
That directory becomes durable output only when Kaggle saves the version/output; it is not Google Drive.
A hard interruption can still lose work not saved by Kaggle. Keep the original Drive copy intact.


The experiment completed on 24 October 2026. This notebook is retained to reproduce or resume that workflow; rerunning it is not required to use the web app. Training source remains pinned to `b9cf2495bfccf3008676ce052f6539b313328d59` so existing results remain compatible. For application installation on Windows, Linux or macOS, see [README](https://github.com/parthpathak719/AI-Detector/blob/main/README.md).


In [ ]:
import os, sys, subprocess, json, hashlib, shutil
from pathlib import Path, PurePosixPath
from zipfile import ZipFile

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HOME'] = '/kaggle/temp/huggingface'
RUN_ROOT = Path('/kaggle/working/muril-run')
REPO = Path('/kaggle/temp/AI-Detector')
SOURCE_COMMIT = 'b9cf2495bfccf3008676ce052f6539b313328d59'
def run(args, **kwargs):
    return subprocess.run([str(x) for x in args], check=True, **kwargs)
def sha(path):
    with path.open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

archives = list(Path('/kaggle/input').rglob('muril-resume.zip'))
indexes = list(Path('/kaggle/input').rglob('transfer_manifest.json'))
assert (len(archives), len(indexes)) in [(1, 0), (0, 1)], 'Attach exactly one private resume dataset.'
z = ZipFile(archives[0]) if archives else None
base = indexes[0].parent if indexes else None
manifest = json.loads(z.read('transfer_manifest.json') if z else indexes[0].read_text())
def copy_checked(name, dst):
    p = PurePosixPath(name)
    assert not p.is_absolute() and '..' not in p.parts
    dst.parent.mkdir(parents=True, exist_ok=True)
    with (z.open(name) if z else (base / name).open('rb')) as src, dst.open('wb') as out:
        shutil.copyfileobj(src, out, 8*1024*1024)
    assert sha(dst) == manifest[name], 'Transfer checksum failed: ' + name

if not RUN_ROOT.exists():
    staging = Path('/kaggle/working/.muril-import')
    if staging.exists(): shutil.rmtree(staging)
    staging.mkdir()
    for name in manifest:
        if name.startswith('run/'):
            copy_checked(name, staging / name[4:])
    (staging / 'import_complete.json').write_text(json.dumps(manifest))
    staging.rename(RUN_ROOT)
assert (RUN_ROOT / 'import_complete.json').is_file(), 'Unrecognized output folder.'
REPO.parent.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    run(['git', 'clone', '--no-checkout', 'https://github.com/parthpathak719/AI-Detector.git', REPO])
run(['git', '-C', REPO, 'fetch', 'origin', SOURCE_COMMIT])
run(['git', '-C', REPO, 'checkout', '--detach', SOURCE_COMMIT])
os.chdir(REPO)
protocol = json.loads((RUN_ROOT / 'reports/protocol.json').read_text())
assert sha(REPO / 'scripts/full_finetune.py') == protocol['script_sha256']
for name in manifest:
    if name.startswith('data/clean_strict/'):
        copy_checked(name, REPO / name)
if z: z.close()

link = REPO / '.cache/colab_local_checkpoints'
destination = RUN_ROOT / 'local_checkpoints'
destination.mkdir(exist_ok=True)
if link.is_symlink():
    assert link.resolve() == destination.resolve()
else:
    assert not link.exists(), 'Unexpected local checkpoints; do not overwrite.'
    link.parent.mkdir(parents=True, exist_ok=True)
    link.symlink_to(destination, target_is_directory=True)
print('Resume files verified. Output directory:', RUN_ROOT)
print('No Drive mounting or Drive writes. One GPU is used.')


## Install an isolated Python 3.12 CUDA environment

This uses uv to provision Python 3.12 independently of Colab's system Python and its optional venv/ensurepip components. It installs CUDA-enabled PyTorch in the isolated environment; no notebook-kernel restart is needed.


In [ ]:
# Colab's system Python may lack ensurepip/venv, or may be Python 3.13.
# Use uv to provision a separate Python 3.12 interpreter and seed pip.
import time
ENVIRONMENT = REPO / '.colab-py312'
PYTHON = ENVIRONMENT / 'bin/python'
probe = subprocess.run(
    [str(PYTHON), '-c', 'import sys, pip; assert sys.version_info[:2] == (3, 12)'],
    capture_output=True, text=True,
) if PYTHON.exists() else None
if probe is None or probe.returncode:
    if ENVIRONMENT.exists():
        backup = ENVIRONMENT.with_name('.colab-py312-incomplete-' + str(time.time_ns()))
        ENVIRONMENT.rename(backup)
        print('Preserved incomplete environment at', backup)
    run([sys.executable, '-m', 'pip', 'install', 'uv==0.7.22'])
    run([sys.executable, '-m', 'uv', 'python', 'install', '3.12'])
    run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', '--seed', ENVIRONMENT])
run([PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip'])
run([PYTHON, '-m', 'pip', 'install', 'torch==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu124'])
run([PYTHON, '-m', 'pip', 'install', '-r', 'requirements.txt', 'gdown==5.2.0'])
# Compatibility workaround for the observed SciPy 1.15.3 L-BFGS native crash.
run([PYTHON, '-m', 'pip', 'install', '--no-deps', 'scipy==1.14.1'])
run([PYTHON, '-c', "import torch; assert torch.cuda.is_available(), 'CUDA unavailable in training environment'; print(torch.__version__, torch.cuda.get_device_name(0))"])


## Validate the imported dataset and download the pinned encoder


In [ ]:
run([PYTHON, '-m', 'scripts.download_muril'])
run([PYTHON, '-X', 'faulthandler', '-u', '-m', 'scripts.check_colab_optimizer'])
check = """
from pathlib import Path
import json
from scripts.train import TASKS, load_task
root = Path('/kaggle/working/muril-run')
protocol = json.loads((root / 'reports/protocol.json').read_text())
counts = {}
for task in TASKS:
    _, digest = load_task(Path('data/clean_strict'), task)
    assert digest == protocol['dataset_sha256'][task], 'Dataset differs: ' + task
    counts[task] = 0
    for variant in ['frozen', 'lr_2e-06', 'lr_5e-06', 'lr_1e-05']:
        for fold in range(5):
            p = root / 'reports' / task / f'{variant}_fold{fold}.json'
            if p.exists():
                assert p.with_suffix('.npz').is_file(), 'Missing predictions: ' + str(p)
                json.loads(p.read_text())
                if variant != 'frozen': counts[task] += 1
            else:
                assert variant != 'frozen', 'Missing frozen baseline'
assert counts['task1_binary'] >= 10 and counts['task2_attribution'] == 15
print('Completed folds to reuse:', counts)
"""
run([PYTHON, '-c', check])


## Resume training
The runner's inherited log text refers to Drive, but RUN_ROOT here is Kaggle output storage.
No Drive files are modified. Do not start a second training session for this run.


In [ ]:
try:
    command = [str(PYTHON), '-X', 'faulthandler', '-u', '-m', 'scripts.colab_run', '--run-root', str(RUN_ROOT), '--recompute-baselines']
    with (RUN_ROOT / 'training.log').open('a', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='')
                log.write(line)
                log.flush()
            returncode = process.wait()
            if returncode == -11:
                raise RuntimeError(
                    "Training hit a native SIGSEGV crash. The faulthandler traceback above "
                    f"and in {RUN_ROOT / 'training.log'} identifies the active Python frames. "
                    "Keep the checkpoints and share that traceback to diagnose the failing library."
                )
            if returncode:
                raise RuntimeError(f"Training stopped with exit code {returncode}. See {RUN_ROOT / 'training.log'}; saved checkpoints remain available.")
        except BaseException:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            raise
except Exception:
    import traceback
    traceback.print_exc()
    print('TRAINING DID NOT FINISH. Preserve this version output and training.log before restarting.')


## Export only after training and model verification complete


In [ ]:
status_file = RUN_ROOT / 'status.json'
status = json.loads(status_file.read_text()) if status_file.exists() else {}
if status.get('phase') == 'complete':
    run([PYTHON, '-m', 'scripts.export_full_run', '--run-root', RUN_ROOT])
    print((RUN_ROOT / 'results.md').read_text())
    print('Download results.zip and selected_models.zip from this version output.')
else:
    print('NOT COMPLETE. Keep the entire muril-run output folder for recovery.')
    print(status)


## Preserve output
After the saved version finishes, open its Output tab. Download muril-run/results.zip for review and
muril-run/selected_models.zip for the Windows app. If training did not finish, preserve the entire
muril-run folder before discarding or replacing this version. Do not publish the dataset or notebook:
resume artifacts include private text examples. Nothing is automatically uploaded to Drive or GitHub.
